In [1]:
"""
================================================================================
SIGN LANGUAGE RECOGNITION – LEAVE-ONE-USER-OUT (LOUO) EVALUATION
================================================================================

This script extracts and structures the per‑class classification reports 
(Precision, Recall, F1‑score, Support) for both TEST and VALIDATION sets 
from three independent LOUO runs (seeds 42, 43, 44) of a sign language 
recognition model (thct_net with palm_ref normalization). 

The original logs are named:
  - run_LOUO_20260706T152636Z.log (seed 42)
  - run_LOUO_20260706T152926Z.log (seed 43)
  - run_LOUO_20260707T035231Z.log (seed 44)

Each run evaluates four users (user1, user2, user3, user5) as the 
left‑out test user, training on the other three. The logs contain 
window‑level classification reports for both the validation and test sets.

PURPOSE:
--------
To create a structured, machine‑readable dataset (CSV files) that preserves 
all per‑class metrics for every fold and seed, allowing further analysis 
(e.g., per‑class performance variability, statistical significance tests, 
or visualisations) in a reproducible manner.

DATA EXTRACTION OVERVIEW:
-------------------------
1. TEST PER-CLASS REPORTS:
   - Extracted from the "TEST WINDOW-LEVEL REPORT" sections of each log.
   - Stored in the nested dictionary `test_data`:
        test_data[seed][test_user][class_name] = [precision, recall, f1, support]
   - Supports are verified to match the known total test samples per fold:
        user1: 38589, user2: 32238, user3: 28025, user5: 37406
     (consistent across all seeds).

2. VALIDATION PER-CLASS REPORTS:
   - Extracted from the "VAL WINDOW-LEVEL REPORT" sections.
   - Stored in `val_classification_data` with the same structure.
   - Support totals verified against known validation set sizes:
        Seed 42:
          user1: 16218, user2: 16967, user3: 16205, user5: 16142
        Seed 43:
          user1: 16201, user2: 16760, user3: 16012, user5: 15965
        Seed 44:
          user1: 16080, user2: 16977, user3: 16087, user5: 16037

3. VALIDATION BEST SCALAR METRICS:
   - Extracted from the final summary lines (e.g., "best_val_acc=0.9965").
   - Stored in `val_scalar_data` as [best_val_acc, best_val_f1] per seed/user.

OUTPUT FILES (CSV):
-------------------
Three CSV files are saved for later use:
1. test_classification_reports.csv
   Columns:
     - seed          : int (42, 43, 44)
     - test_user     : str ('user1'–'user5')
     - class_name    : str (sign gloss or 'background')
     - precision     : float
     - recall        : float
     - f1            : float
     - support       : int (number of test windows per class)

2. val_classification_reports.csv
   Same schema as above, but for the validation set.

3. val_metrics.csv
   Columns:
     - seed          : int
     - test_user     : str
     - best_val_acc  : float (best validation accuracy on the selected model)
     - best_val_f1   : float (best validation macro‑averaged F1)

VERIFICATION:
-------------
- The script sums the `support` column per (seed, test_user) for both test 
  and validation DataFrames and compares them with the known totals.
- All sums match, confirming that no class or value was omitted during 
  extraction.

CALCULATED STATISTICS:
---------------------
The script automatically computes and prints:
1. For the TEST set:
   - Per‑user (across 3 seeds) mean and standard deviation of:
        * overall accuracy (directly from log summary)
        * macro‑averaged F1 (mean of per‑class F1 scores)
   - Overall (across all users and seeds) mean and standard deviation.

2. For the VALIDATION set:
   - Per‑user mean and std of the best_val_acc and best_val_f1.
   - Overall mean and std.

These statistics match the manually verified numbers from the thesis.

RECOMMENDATIONS FOR FURTHER AI PROCESSING:
------------------------------------------
- The CSV files are self‑contained and can be loaded with pandas for:
    * Computing class‑wise averages across seeds for each user.
    * Comparing performance between users (e.g., user1 vs. user5).
    * Analysing the variability of specific glosses (e.g., "COME").
    * Testing statistical significance (paired t‑tests) between models 
      if multiple model variants are available.
- The `seed` column allows for reproducibility and can be used as a 
  grouping variable for uncertainty estimation.
- For any new metric (e.g., weighted F1, Matthews correlation), you can 
  derive it from the raw per‑class counts by weighting the F1 scores 
  by support.

ASSUMPTIONS & LIMITATIONS:
--------------------------
- The test set accuracy is taken directly from the log's "Accuracy:" line, 
  not recomputed from the per‑class data (to avoid rounding discrepancies).
- The validation best metrics (`best_val_acc` and `best_val_f1`) are the 
  scalar values saved by the training script; they correspond to the 
  model selected at the end of training.
- The per‑class reports are from the **best model** on the validation set, 
  not from the final epoch, because the logs state "best model restored from memory".
- The background class is included in all reports; it can be excluded 
  if only sign glosses are of interest.
- All metrics are based on **window‑level** classification (sliding windows), 
  not sequence‑level (WER) evaluation.

USAGE EXAMPLE FOR FUTURE ANALYSIS:
----------------------------------
>>> import pandas as pd
>>> test_df = pd.read_csv('test_classification_reports.csv')
>>> val_df = pd.read_csv('val_classification_reports.csv')
>>> 
>>> # Compute per‑class average F1 across all seeds and users
>>> avg_f1_per_class = test_df.groupby('class_name')['f1'].mean().sort_values()
>>> 
>>> # Get user2 performance on 'COME' across seeds
>>> user2_come = test_df[(test_df['test_user']=='user2') & (test_df['class_name']=='COME')]
>>> print(user2_come[['seed', 'precision', 'recall', 'f1']])
>>> 
>>> # Load validation scalar metrics
>>> val_scalar = pd.read_csv('val_metrics.csv')
>>> val_scalar.groupby('test_user').agg({'best_val_acc': ['mean','std']})

AUTHOR NOTE:
------------
This script was generated on 2026-07-08 based on manual extraction from logs.
All numbers have been cross‑verified against the original log files to 
ensure correctness.
================================================================================
"""

import pandas as pd
import numpy as np
from io import StringIO

# =============================================================================
# 1. HARD-CODED DATA EXTRACTED FROM YOUR LOGS
# =============================================================================

# ---- Test Classification Reports (per seed, per test_user, per class) ----
# Structure: data[seed][test_user][class_name] = [precision, recall, f1, support]
test_data = {
    42: {
        'user1': {
            'AUGUST': [0.84, 1.00, 0.91, 971],
            'BIG': [0.95, 0.91, 0.93, 655],
            'BIRD': [0.95, 0.88, 0.91, 892],
            'BOAT': [0.84, 0.98, 0.91, 589],
            'COME': [0.94, 0.55, 0.70, 779],
            'DRIVER': [0.95, 0.97, 0.96, 1250],
            'FARMING': [0.98, 0.83, 0.90, 1257],
            'FEBRUARY': [0.87, 0.95, 0.91, 994],
            'GO': [0.71, 0.68, 0.69, 465],
            'GREETINGS': [1.00, 0.86, 0.92, 846],
            'OUR': [0.79, 0.91, 0.85, 591],
            'READ': [1.00, 0.83, 0.91, 1256],
            'SMALL': [0.66, 1.00, 0.79, 815],
            'TIGER': [0.91, 1.00, 0.95, 657],
            'TRAIN': [0.90, 0.79, 0.84, 880],
            'UGLY': [0.84, 0.81, 0.82, 779],
            'VAN': [0.92, 0.97, 0.94, 828],
            'WHAT': [0.75, 0.95, 0.84, 75],
            'WHICH': [0.79, 0.53, 0.63, 884],
            'WRITE': [1.00, 1.00, 1.00, 422],
            'background': [0.98, 0.99, 0.98, 22704],
        },
        'user2': {
            'AUGUST': [0.97, 1.00, 0.99, 978],
            'BIG': [0.81, 0.81, 0.81, 465],
            'BIRD': [0.96, 1.00, 0.98, 699],
            'BOAT': [0.88, 1.00, 0.93, 357],
            'COME': [0.94, 0.82, 0.88, 228],
            'DRIVER': [1.00, 1.00, 1.00, 1210],
            'FARMING': [0.98, 0.99, 0.98, 510],
            'FEBRUARY': [1.00, 0.64, 0.78, 1039],
            'GO': [0.63, 0.93, 0.75, 172],
            'GREETINGS': [0.62, 1.00, 0.76, 673],
            'OUR': [0.87, 0.99, 0.93, 275],
            'READ': [0.99, 1.00, 1.00, 1178],
            'SMALL': [0.83, 0.99, 0.90, 504],
            'TIGER': [1.00, 0.67, 0.80, 241],
            'TRAIN': [0.99, 1.00, 1.00, 428],
            'UGLY': [0.96, 0.80, 0.87, 299],
            'VAN': [1.00, 1.00, 1.00, 216],
            'WHAT': [1.00, 1.00, 1.00, 18],
            'WHICH': [0.98, 0.95, 0.96, 754],
            'WRITE': [1.00, 1.00, 1.00, 605],
            'background': [1.00, 0.99, 0.99, 21389],
        },
        'user3': {
            'AUGUST': [1.00, 1.00, 1.00, 845],
            'BIG': [1.00, 0.95, 0.98, 371],
            'BIRD': [1.00, 1.00, 1.00, 555],
            'BOAT': [0.99, 0.75, 0.86, 490],
            'COME': [0.93, 1.00, 0.97, 331],
            'DRIVER': [1.00, 1.00, 1.00, 1005],
            'FARMING': [0.99, 0.95, 0.97, 634],
            'FEBRUARY': [0.66, 0.99, 0.79, 746],
            'GO': [1.00, 1.00, 1.00, 91],
            'GREETINGS': [1.00, 0.39, 0.56, 597],
            'OUR': [0.88, 0.85, 0.87, 281],
            'READ': [0.89, 1.00, 0.94, 756],
            'SMALL': [0.95, 0.99, 0.97, 575],
            'TIGER': [1.00, 0.98, 0.99, 332],
            'TRAIN': [0.90, 1.00, 0.95, 446],
            'UGLY': [0.93, 0.93, 0.93, 451],
            'VAN': [0.93, 0.94, 0.94, 500],
            'WHAT': [0.88, 0.49, 0.63, 73],
            'WHICH': [0.99, 1.00, 1.00, 523],
            'WRITE': [0.96, 1.00, 0.98, 798],
            'background': [1.00, 1.00, 1.00, 17625],
        },
        'user5': {
            'AUGUST': [1.00, 1.00, 1.00, 1049],
            'BIG': [0.99, 1.00, 1.00, 314],
            'BIRD': [0.98, 1.00, 0.99, 508],
            'BOAT': [0.92, 1.00, 0.96, 463],
            'COME': [0.94, 0.94, 0.94, 586],
            'DRIVER': [1.00, 0.99, 1.00, 1150],
            'FARMING': [0.98, 0.97, 0.98, 368],
            'FEBRUARY': [0.97, 1.00, 0.98, 831],
            'GO': [0.84, 0.77, 0.80, 349],
            'GREETINGS': [1.00, 0.92, 0.96, 766],
            'OUR': [0.84, 0.75, 0.79, 260],
            'READ': [1.00, 0.95, 0.97, 818],
            'SMALL': [0.90, 0.97, 0.93, 622],
            'TIGER': [0.98, 1.00, 0.99, 463],
            'TRAIN': [0.97, 1.00, 0.99, 756],
            'UGLY': [0.74, 0.87, 0.80, 494],
            'VAN': [0.82, 1.00, 0.90, 323],
            'WHAT': [0.61, 1.00, 0.75, 66],
            'WHICH': [0.99, 0.89, 0.94, 608],
            'WRITE': [1.00, 0.94, 0.97, 964],
            'background': [1.00, 0.99, 1.00, 25648],
        },
    },
    43: {
        'user1': {
            'AUGUST': [1.00, 0.94, 0.97, 971],
            'BIG': [0.98, 0.93, 0.95, 655],
            'BIRD': [0.74, 0.99, 0.85, 892],
            'BOAT': [0.91, 1.00, 0.95, 589],
            'COME': [0.96, 0.32, 0.48, 779],
            'DRIVER': [0.91, 0.96, 0.93, 1250],
            'FARMING': [0.98, 0.75, 0.85, 1257],
            'FEBRUARY': [0.80, 0.95, 0.87, 994],
            'GO': [0.99, 0.64, 0.78, 465],
            'GREETINGS': [1.00, 0.72, 0.84, 846],
            'OUR': [0.53, 0.91, 0.67, 591],
            'READ': [1.00, 0.86, 0.93, 1256],
            'SMALL': [0.90, 0.99, 0.95, 815],
            'TIGER': [0.86, 0.72, 0.78, 657],
            'TRAIN': [0.95, 0.71, 0.81, 880],
            'UGLY': [0.69, 0.92, 0.79, 779],
            'VAN': [0.79, 0.98, 0.88, 828],
            'WHAT': [0.41, 1.00, 0.59, 75],
            'WHICH': [0.79, 0.57, 0.66, 884],
            'WRITE': [0.83, 1.00, 0.91, 422],
            'background': [0.98, 0.99, 0.99, 22704],
        },
        'user2': {
            'AUGUST': [0.97, 1.00, 0.99, 978],
            'BIG': [0.91, 0.77, 0.83, 465],
            'BIRD': [1.00, 0.67, 0.80, 699],
            'BOAT': [0.93, 1.00, 0.96, 357],
            'COME': [0.81, 0.89, 0.85, 228],
            'DRIVER': [1.00, 1.00, 1.00, 1210],
            'FARMING': [1.00, 0.99, 1.00, 510],
            'FEBRUARY': [0.97, 0.87, 0.91, 1039],
            'GO': [0.59, 0.72, 0.64, 172],
            'GREETINGS': [0.80, 1.00, 0.89, 673],
            'OUR': [0.92, 0.97, 0.95, 275],
            'READ': [0.99, 1.00, 1.00, 1178],
            'SMALL': [0.80, 1.00, 0.89, 504],
            'TIGER': [1.00, 1.00, 1.00, 241],
            'TRAIN': [1.00, 1.00, 1.00, 428],
            'UGLY': [0.98, 0.74, 0.84, 299],
            'VAN': [1.00, 1.00, 1.00, 216],
            'WHAT': [1.00, 1.00, 1.00, 18],
            'WHICH': [1.00, 0.96, 0.98, 754],
            'WRITE': [1.00, 1.00, 1.00, 605],
            'background': [0.99, 0.99, 0.99, 21389],
        },
        'user3': {
            'AUGUST': [1.00, 0.99, 0.99, 845],
            'BIG': [1.00, 1.00, 1.00, 371],
            'BIRD': [0.97, 1.00, 0.98, 555],
            'BOAT': [1.00, 0.83, 0.91, 490],
            'COME': [0.99, 0.99, 0.99, 331],
            'DRIVER': [1.00, 1.00, 1.00, 1005],
            'FARMING': [0.96, 0.97, 0.96, 634],
            'FEBRUARY': [0.79, 1.00, 0.88, 746],
            'GO': [1.00, 1.00, 1.00, 91],
            'GREETINGS': [1.00, 0.82, 0.90, 597],
            'OUR': [1.00, 0.33, 0.50, 281],
            'READ': [0.88, 1.00, 0.93, 756],
            'SMALL': [1.00, 1.00, 1.00, 575],
            'TIGER': [1.00, 1.00, 1.00, 332],
            'TRAIN': [0.72, 1.00, 0.84, 446],
            'UGLY': [0.93, 0.78, 0.85, 451],
            'VAN': [1.00, 0.97, 0.98, 500],
            'WHAT': [0.00, 0.00, 0.00, 73],
            'WHICH': [0.99, 1.00, 1.00, 523],
            'WRITE': [0.99, 1.00, 1.00, 798],
            'background': [1.00, 1.00, 1.00, 17625],
        },
        'user5': {
            'AUGUST': [0.99, 1.00, 0.99, 1049],
            'BIG': [0.82, 1.00, 0.90, 314],
            'BIRD': [0.96, 1.00, 0.98, 508],
            'BOAT': [0.88, 1.00, 0.93, 463],
            'COME': [0.88, 1.00, 0.94, 586],
            'DRIVER': [1.00, 0.99, 1.00, 1150],
            'FARMING': [0.82, 0.96, 0.89, 368],
            'FEBRUARY': [1.00, 1.00, 1.00, 831],
            'GO': [0.95, 0.39, 0.55, 349],
            'GREETINGS': [1.00, 1.00, 1.00, 766],
            'OUR': [0.62, 0.82, 0.71, 260],
            'READ': [1.00, 0.92, 0.96, 818],
            'SMALL': [1.00, 0.73, 0.84, 622],
            'TIGER': [1.00, 1.00, 1.00, 463],
            'TRAIN': [0.87, 1.00, 0.93, 756],
            'UGLY': [0.80, 0.84, 0.82, 494],
            'VAN': [0.95, 1.00, 0.98, 323],
            'WHAT': [0.97, 1.00, 0.99, 66],
            'WHICH': [0.95, 0.97, 0.96, 608],
            'WRITE': [0.99, 1.00, 1.00, 964],
            'background': [0.99, 0.99, 0.99, 25648],
        },
    },
    44: {
        'user1': {
            'AUGUST': [0.96, 1.00, 0.98, 971],
            'BIG': [0.97, 0.90, 0.93, 655],
            'BIRD': [0.93, 0.91, 0.92, 892],
            'BOAT': [0.95, 1.00, 0.98, 589],
            'COME': [0.97, 0.44, 0.60, 779],
            'DRIVER': [1.00, 0.96, 0.98, 1250],
            'FARMING': [0.98, 0.85, 0.91, 1257],
            'FEBRUARY': [0.76, 0.96, 0.85, 994],
            'GO': [0.94, 0.70, 0.80, 465],
            'GREETINGS': [1.00, 0.64, 0.78, 846],
            'OUR': [0.60, 0.97, 0.75, 591],
            'READ': [1.00, 0.93, 0.96, 1256],
            'SMALL': [0.75, 1.00, 0.86, 815],
            'TIGER': [0.81, 0.99, 0.89, 657],
            'TRAIN': [0.96, 0.66, 0.78, 880],
            'UGLY': [0.82, 0.95, 0.88, 779],
            'VAN': [0.88, 0.97, 0.92, 828],
            'WHAT': [0.60, 1.00, 0.75, 75],
            'WHICH': [0.91, 0.59, 0.72, 884],
            'WRITE': [1.00, 1.00, 1.00, 422],
            'background': [0.97, 0.99, 0.98, 22704],
        },
        'user2': {
            'AUGUST': [1.00, 1.00, 1.00, 978],
            'BIG': [0.87, 0.91, 0.89, 465],
            'BIRD': [1.00, 0.77, 0.87, 699],
            'BOAT': [0.95, 1.00, 0.97, 357],
            'COME': [0.88, 0.88, 0.88, 228],
            'DRIVER': [1.00, 1.00, 1.00, 1210],
            'FARMING': [0.98, 1.00, 0.99, 510],
            'FEBRUARY': [1.00, 0.99, 1.00, 1039],
            'GO': [0.75, 0.85, 0.80, 172],
            'GREETINGS': [0.99, 1.00, 1.00, 673],
            'OUR': [0.92, 0.95, 0.94, 275],
            'READ': [1.00, 1.00, 1.00, 1178],
            'SMALL': [0.91, 0.98, 0.94, 504],
            'TIGER': [1.00, 1.00, 1.00, 241],
            'TRAIN': [1.00, 0.99, 1.00, 428],
            'UGLY': [0.99, 0.76, 0.86, 299],
            'VAN': [1.00, 1.00, 1.00, 216],
            'WHAT': [1.00, 1.00, 1.00, 18],
            'WHICH': [0.98, 0.99, 0.98, 754],
            'WRITE': [1.00, 1.00, 1.00, 605],
            'background': [0.99, 0.99, 0.99, 21389],
        },
        'user3': {
            'AUGUST': [1.00, 1.00, 1.00, 845],
            'BIG': [1.00, 1.00, 1.00, 371],
            'BIRD': [1.00, 1.00, 1.00, 555],
            'BOAT': [1.00, 0.66, 0.80, 490],
            'COME': [0.94, 0.96, 0.95, 331],
            'DRIVER': [1.00, 1.00, 1.00, 1005],
            'FARMING': [0.97, 0.93, 0.95, 634],
            'FEBRUARY': [0.58, 1.00, 0.73, 746],
            'GO': [0.97, 1.00, 0.98, 91],
            'GREETINGS': [1.00, 0.18, 0.31, 597],
            'OUR': [0.87, 0.97, 0.92, 281],
            'READ': [0.84, 1.00, 0.91, 756],
            'SMALL': [1.00, 1.00, 1.00, 575],
            'TIGER': [1.00, 1.00, 1.00, 332],
            'TRAIN': [1.00, 1.00, 1.00, 446],
            'UGLY': [0.71, 0.86, 0.78, 451],
            'VAN': [0.95, 1.00, 0.97, 500],
            'WHAT': [1.00, 0.82, 0.90, 73],
            'WHICH': [1.00, 1.00, 1.00, 523],
            'WRITE': [1.00, 1.00, 1.00, 798],
            'background': [1.00, 0.99, 0.99, 17625],
        },
        'user5': {
            'AUGUST': [1.00, 1.00, 1.00, 1049],
            'BIG': [1.00, 1.00, 1.00, 314],
            'BIRD': [0.98, 1.00, 0.99, 508],
            'BOAT': [0.95, 1.00, 0.98, 463],
            'COME': [0.87, 1.00, 0.93, 586],
            'DRIVER': [1.00, 1.00, 1.00, 1150],
            'FARMING': [0.92, 0.96, 0.94, 368],
            'FEBRUARY': [0.98, 1.00, 0.99, 831],
            'GO': [0.97, 0.54, 0.70, 349],
            'GREETINGS': [0.99, 0.99, 0.99, 766],
            'OUR': [0.85, 0.74, 0.79, 260],
            'READ': [1.00, 0.97, 0.99, 818],
            'SMALL': [0.91, 0.98, 0.95, 622],
            'TIGER': [0.99, 1.00, 1.00, 463],
            'TRAIN': [0.99, 0.98, 0.98, 756],
            'UGLY': [0.74, 0.98, 0.84, 494],
            'VAN': [0.96, 1.00, 0.98, 323],
            'WHAT': [0.85, 1.00, 0.92, 66],
            'WHICH': [0.89, 0.98, 0.93, 608],
            'WRITE': [1.00, 1.00, 1.00, 964],
            'background': [1.00, 0.99, 0.99, 25648],
        },
    },
}


# ---- VALIDATION Classification Reports (per seed, per test_user, per class) ----
val_classification_data = {
    42: {
        'user1': {
            'AUGUST': [0.99, 1.00, 1.00, 534],
            'BIG': [1.00, 1.00, 1.00, 150],
            'BIRD': [1.00, 1.00, 1.00, 266],
            'BOAT': [1.00, 1.00, 1.00, 229],
            'COME': [1.00, 0.98, 0.99, 179],
            'DRIVER': [1.00, 1.00, 1.00, 548],
            'FARMING': [1.00, 0.90, 0.95, 238],
            'FEBRUARY': [1.00, 1.00, 1.00, 437],
            'GO': [0.84, 1.00, 0.91, 116],
            'GREETINGS': [1.00, 1.00, 1.00, 379],
            'OUR': [0.98, 1.00, 0.99, 151],
            'READ': [1.00, 1.00, 1.00, 409],
            'SMALL': [1.00, 1.00, 1.00, 258],
            'TIGER': [1.00, 1.00, 1.00, 182],
            'TRAIN': [1.00, 1.00, 1.00, 280],
            'UGLY': [1.00, 0.93, 0.96, 243],
            'VAN': [0.95, 1.00, 0.97, 185],
            'WHAT': [1.00, 1.00, 1.00, 15],
            'WHICH': [1.00, 1.00, 1.00, 373],
            'WRITE': [1.00, 1.00, 1.00, 369],
            'background': [1.00, 1.00, 1.00, 10677],
        },
        'user2': {
            'AUGUST': [0.98, 1.00, 0.99, 510],
            'BIG': [1.00, 1.00, 1.00, 185],
            'BIRD': [0.95, 1.00, 0.98, 296],
            'BOAT': [0.95, 1.00, 0.98, 267],
            'COME': [1.00, 1.00, 1.00, 269],
            'DRIVER': [1.00, 1.00, 1.00, 546],
            'FARMING': [0.99, 0.92, 0.96, 302],
            'FEBRUARY': [1.00, 1.00, 1.00, 426],
            'GO': [0.89, 1.00, 0.94, 158],
            'GREETINGS': [1.00, 1.00, 1.00, 373],
            'OUR': [0.94, 1.00, 0.97, 206],
            'READ': [1.00, 1.00, 1.00, 458],
            'SMALL': [1.00, 1.00, 1.00, 304],
            'TIGER': [1.00, 1.00, 1.00, 236],
            'TRAIN': [0.99, 1.00, 0.99, 327],
            'UGLY': [0.99, 0.96, 0.98, 302],
            'VAN': [0.95, 1.00, 0.97, 251],
            'WHAT': [1.00, 1.00, 1.00, 28],
            'WHICH': [1.00, 1.00, 1.00, 335],
            'WRITE': [1.00, 1.00, 1.00, 347],
            'background': [1.00, 0.99, 1.00, 10841],
        },
        'user3': {
            'AUGUST': [0.99, 1.00, 0.99, 485],
            'BIG': [1.00, 1.00, 1.00, 167],
            'BIRD': [0.95, 1.00, 0.97, 303],
            'BOAT': [0.98, 1.00, 0.99, 228],
            'COME': [0.97, 1.00, 0.99, 227],
            'DRIVER': [1.00, 1.00, 1.00, 587],
            'FARMING': [1.00, 1.00, 1.00, 227],
            'FEBRUARY': [1.00, 0.96, 0.98, 433],
            'GO': [1.00, 0.96, 0.98, 171],
            'GREETINGS': [1.00, 1.00, 1.00, 432],
            'OUR': [0.93, 1.00, 0.96, 160],
            'READ': [1.00, 1.00, 1.00, 505],
            'SMALL': [1.00, 1.00, 1.00, 274],
            'TIGER': [0.98, 1.00, 0.99, 213],
            'TRAIN': [1.00, 1.00, 1.00, 287],
            'UGLY': [0.91, 0.99, 0.95, 253],
            'VAN': [1.00, 1.00, 1.00, 191],
            'WHAT': [1.00, 1.00, 1.00, 18],
            'WHICH': [0.99, 1.00, 0.99, 390],
            'WRITE': [1.00, 1.00, 1.00, 284],
            'background': [1.00, 0.99, 1.00, 10370],
        },
        'user5': {
            'AUGUST': [1.00, 1.00, 1.00, 502],
            'BIG': [1.00, 1.00, 1.00, 191],
            'BIRD': [0.99, 1.00, 1.00, 311],
            'BOAT': [0.97, 1.00, 0.99, 230],
            'COME': [0.99, 0.98, 0.98, 228],
            'DRIVER': [1.00, 1.00, 1.00, 527],
            'FARMING': [0.99, 0.92, 0.96, 304],
            'FEBRUARY': [1.00, 0.98, 0.99, 456],
            'GO': [1.00, 1.00, 1.00, 140],
            'GREETINGS': [1.00, 1.00, 1.00, 397],
            'OUR': [0.99, 0.99, 0.99, 200],
            'READ': [1.00, 1.00, 1.00, 473],
            'SMALL': [1.00, 1.00, 1.00, 280],
            'TIGER': [1.00, 1.00, 1.00, 206],
            'TRAIN': [1.00, 1.00, 1.00, 291],
            'UGLY': [0.96, 1.00, 0.98, 273],
            'VAN': [0.91, 1.00, 0.96, 234],
            'WHAT': [1.00, 1.00, 1.00, 23],
            'WHICH': [1.00, 1.00, 1.00, 384],
            'WRITE': [1.00, 1.00, 1.00, 314],
            'background': [1.00, 1.00, 1.00, 10178],
        },
    },
    43: {
        'user1': {
            'AUGUST': [1.00, 1.00, 1.00, 579],
            'BIG': [1.00, 1.00, 1.00, 175],
            'BIRD': [1.00, 1.00, 1.00, 296],
            'BOAT': [1.00, 1.00, 1.00, 181],
            'COME': [1.00, 1.00, 1.00, 162],
            'DRIVER': [1.00, 1.00, 1.00, 586],
            'FARMING': [1.00, 1.00, 1.00, 209],
            'FEBRUARY': [1.00, 1.00, 1.00, 452],
            'GO': [0.95, 1.00, 0.97, 117],
            'GREETINGS': [1.00, 1.00, 1.00, 336],
            'OUR': [1.00, 1.00, 1.00, 162],
            'READ': [1.00, 1.00, 1.00, 497],
            'SMALL': [1.00, 1.00, 1.00, 269],
            'TIGER': [1.00, 1.00, 1.00, 196],
            'TRAIN': [1.00, 1.00, 1.00, 249],
            'UGLY': [0.97, 1.00, 0.99, 219],
            'VAN': [1.00, 1.00, 1.00, 164],
            'WHAT': [0.95, 1.00, 0.97, 39],
            'WHICH': [1.00, 0.96, 0.98, 292],
            'WRITE': [1.00, 1.00, 1.00, 481],
            'background': [1.00, 1.00, 1.00, 10540],
        },
        'user2': {
            'AUGUST': [1.00, 1.00, 1.00, 540],
            'BIG': [1.00, 1.00, 1.00, 212],
            'BIRD': [1.00, 1.00, 1.00, 313],
            'BOAT': [1.00, 1.00, 1.00, 236],
            'COME': [1.00, 1.00, 1.00, 243],
            'DRIVER': [1.00, 1.00, 1.00, 571],
            'FARMING': [0.98, 0.92, 0.95, 396],
            'FEBRUARY': [1.00, 1.00, 1.00, 410],
            'GO': [0.95, 1.00, 0.97, 153],
            'GREETINGS': [1.00, 1.00, 1.00, 340],
            'OUR': [0.99, 1.00, 1.00, 211],
            'READ': [1.00, 1.00, 1.00, 375],
            'SMALL': [1.00, 1.00, 1.00, 304],
            'TIGER': [0.98, 1.00, 0.99, 219],
            'TRAIN': [1.00, 1.00, 1.00, 353],
            'UGLY': [0.98, 1.00, 0.99, 304],
            'VAN': [1.00, 0.97, 0.99, 274],
            'WHAT': [0.95, 1.00, 0.97, 53],
            'WHICH': [0.90, 1.00, 0.95, 326],
            'WRITE': [1.00, 1.00, 1.00, 408],
            'background': [1.00, 1.00, 1.00, 10519],
        },
        'user3': {
            'AUGUST': [1.00, 1.00, 1.00, 547],
            'BIG': [1.00, 0.99, 1.00, 234],
            'BIRD': [1.00, 1.00, 1.00, 317],
            'BOAT': [1.00, 1.00, 1.00, 198],
            'COME': [1.00, 1.00, 1.00, 213],
            'DRIVER': [1.00, 1.00, 1.00, 537],
            'FARMING': [0.95, 0.91, 0.93, 362],
            'FEBRUARY': [1.00, 0.98, 0.99, 443],
            'GO': [0.98, 1.00, 0.99, 166],
            'GREETINGS': [1.00, 1.00, 1.00, 336],
            'OUR': [1.00, 0.99, 1.00, 176],
            'READ': [1.00, 1.00, 1.00, 448],
            'SMALL': [1.00, 1.00, 1.00, 273],
            'TIGER': [0.98, 1.00, 0.99, 206],
            'TRAIN': [1.00, 1.00, 1.00, 285],
            'UGLY': [0.92, 1.00, 0.96, 249],
            'VAN': [1.00, 0.91, 0.95, 203],
            'WHAT': [1.00, 1.00, 1.00, 34],
            'WHICH': [0.90, 1.00, 0.95, 292],
            'WRITE': [1.00, 1.00, 1.00, 369],
            'background': [1.00, 1.00, 1.00, 10124],
        },
        'user5': {
            'AUGUST': [1.00, 1.00, 1.00, 521],
            'BIG': [0.98, 1.00, 0.99, 234],
            'BIRD': [1.00, 1.00, 1.00, 376],
            'BOAT': [1.00, 1.00, 1.00, 210],
            'COME': [1.00, 1.00, 1.00, 195],
            'DRIVER': [1.00, 1.00, 1.00, 628],
            'FARMING': [0.95, 0.93, 0.94, 410],
            'FEBRUARY': [1.00, 0.99, 0.99, 477],
            'GO': [1.00, 1.00, 1.00, 134],
            'GREETINGS': [1.00, 1.00, 1.00, 320],
            'OUR': [0.99, 1.00, 1.00, 216],
            'READ': [1.00, 1.00, 1.00, 444],
            'SMALL': [1.00, 0.98, 0.99, 270],
            'TIGER': [0.98, 1.00, 0.99, 171],
            'TRAIN': [1.00, 1.00, 1.00, 292],
            'UGLY': [0.98, 1.00, 0.99, 278],
            'VAN': [1.00, 0.91, 0.95, 241],
            'WHAT': [0.95, 1.00, 0.98, 42],
            'WHICH': [0.91, 1.00, 0.96, 308],
            'WRITE': [1.00, 1.00, 1.00, 383],
            'background': [1.00, 1.00, 1.00, 9815],
        },
    },
    44: {
        'user1': {
            'AUGUST': [1.00, 1.00, 1.00, 487],
            'BIG': [0.91, 1.00, 0.95, 195],
            'BIRD': [1.00, 1.00, 1.00, 281],
            'BOAT': [1.00, 1.00, 1.00, 231],
            'COME': [1.00, 1.00, 1.00, 170],
            'DRIVER': [1.00, 1.00, 1.00, 649],
            'FARMING': [1.00, 0.99, 1.00, 252],
            'FEBRUARY': [1.00, 0.99, 1.00, 437],
            'GO': [0.90, 1.00, 0.95, 94],
            'GREETINGS': [1.00, 1.00, 1.00, 269],
            'OUR': [1.00, 0.98, 0.99, 129],
            'READ': [1.00, 1.00, 1.00, 476],
            'SMALL': [0.91, 1.00, 0.95, 267],
            'TIGER': [0.93, 1.00, 0.96, 193],
            'TRAIN': [1.00, 1.00, 1.00, 234],
            'UGLY': [0.98, 1.00, 0.99, 225],
            'VAN': [0.99, 1.00, 0.99, 171],
            'WHAT': [1.00, 1.00, 1.00, 40],
            'WHICH': [0.97, 0.97, 0.97, 264],
            'WRITE': [1.00, 1.00, 1.00, 381],
            'background': [1.00, 0.99, 1.00, 10635],
        },
        'user2': {
            'AUGUST': [1.00, 1.00, 1.00, 509],
            'BIG': [1.00, 1.00, 1.00, 178],
            'BIRD': [1.00, 1.00, 1.00, 336],
            'BOAT': [1.00, 1.00, 1.00, 238],
            'COME': [0.93, 1.00, 0.97, 236],
            'DRIVER': [1.00, 1.00, 1.00, 546],
            'FARMING': [1.00, 1.00, 1.00, 325],
            'FEBRUARY': [0.97, 1.00, 0.98, 456],
            'GO': [0.90, 1.00, 0.95, 114],
            'GREETINGS': [1.00, 1.00, 1.00, 351],
            'OUR': [1.00, 0.92, 0.96, 196],
            'READ': [1.00, 1.00, 1.00, 428],
            'SMALL': [0.96, 1.00, 0.98, 307],
            'TIGER': [0.98, 1.00, 0.99, 241],
            'TRAIN': [0.98, 1.00, 0.99, 329],
            'UGLY': [0.98, 0.95, 0.96, 288],
            'VAN': [1.00, 1.00, 1.00, 230],
            'WHAT': [1.00, 0.98, 0.99, 64],
            'WHICH': [0.97, 0.92, 0.95, 375],
            'WRITE': [1.00, 1.00, 1.00, 363],
            'background': [1.00, 1.00, 1.00, 10867],
        },
        'user3': {
            'AUGUST': [1.00, 1.00, 1.00, 489],
            'BIG': [0.91, 1.00, 0.95, 206],
            'BIRD': [1.00, 1.00, 1.00, 305],
            'BOAT': [1.00, 1.00, 1.00, 210],
            'COME': [0.98, 1.00, 0.99, 207],
            'DRIVER': [1.00, 1.00, 1.00, 589],
            'FARMING': [1.00, 0.98, 0.99, 320],
            'FEBRUARY': [0.95, 1.00, 0.97, 460],
            'GO': [0.94, 1.00, 0.97, 127],
            'GREETINGS': [1.00, 1.00, 1.00, 298],
            'OUR': [0.95, 0.88, 0.91, 165],
            'READ': [1.00, 1.00, 1.00, 480],
            'SMALL': [0.91, 0.99, 0.94, 282],
            'TIGER': [0.98, 1.00, 0.99, 195],
            'TRAIN': [0.99, 1.00, 0.99, 251],
            'UGLY': [0.91, 0.89, 0.90, 245],
            'VAN': [0.96, 1.00, 0.98, 162],
            'WHAT': [1.00, 1.00, 1.00, 45],
            'WHICH': [0.96, 0.95, 0.96, 329],
            'WRITE': [1.00, 1.00, 1.00, 326],
            'background': [1.00, 0.99, 1.00, 10396],
        },
        'user5': {
            'AUGUST': [1.00, 1.00, 1.00, 489],
            'BIG': [0.89, 1.00, 0.94, 210],
            'BIRD': [1.00, 1.00, 1.00, 359],
            'BOAT': [1.00, 1.00, 1.00, 257],
            'COME': [0.85, 1.00, 0.92, 203],
            'DRIVER': [1.00, 1.00, 1.00, 697],
            'FARMING': [1.00, 0.99, 1.00, 351],
            'FEBRUARY': [0.95, 1.00, 0.98, 489],
            'GO': [1.00, 0.97, 0.98, 94],
            'GREETINGS': [1.00, 1.00, 1.00, 309],
            'OUR': [1.00, 0.90, 0.95, 197],
            'READ': [1.00, 1.00, 1.00, 500],
            'SMALL': [0.91, 1.00, 0.95, 248],
            'TIGER': [0.97, 1.00, 0.98, 175],
            'TRAIN': [1.00, 1.00, 1.00, 266],
            'UGLY': [1.00, 0.90, 0.95, 262],
            'VAN': [1.00, 1.00, 1.00, 190],
            'WHAT': [1.00, 1.00, 1.00, 52],
            'WHICH': [1.00, 0.96, 0.98, 331],
            'WRITE': [1.00, 1.00, 1.00, 304],
            'background': [1.00, 0.99, 1.00, 10054],
        },
    },
}


# ---- Validation Best Metrics (per seed, per test_user) ----
val_scalar_data = {
    42: {
        'user1': [0.996547, 0.989321],
        'user2': [0.994047, 0.988430],
        'user3': [0.995063, 0.990368],
        'user5': [0.996531, 0.991886],
    },
    43: {
        'user1': [0.998457, 0.996005],
        'user2': [0.996122, 0.990803],
        'user3': [0.994754, 0.987820],
        'user5': [0.995177, 0.988758],
    },
    44: {
        'user1': [0.994714, 0.988287],
        'user2': [0.994758, 0.986493],
        'user3': [0.990924, 0.978589],
        'user5': [0.992580, 0.982156],
    },
}


# =============================================================================
# 2. BUILD PANDAS DATAFRAMES
# =============================================================================

# 2a. Test per-class reports
test_records = []
for seed, users in test_data.items():
    for user, classes in users.items():
        for cls, metrics in classes.items():
            test_records.append({
                'seed': seed,
                'test_user': user,
                'class_name': cls,
                'precision': metrics[0],
                'recall': metrics[1],
                'f1': metrics[2],
                'support': metrics[3],
            })
test_df = pd.DataFrame(test_records)

# 2b. Validation per-class reports
val_records = []
for seed, users in val_classification_data.items():
    for user, classes in users.items():
        for cls, metrics in classes.items():
            val_records.append({
                'seed': seed,
                'test_user': user,
                'class_name': cls,
                'precision': metrics[0],
                'recall': metrics[1],
                'f1': metrics[2],
                'support': metrics[3],
            })
val_df = pd.DataFrame(val_records)

# 2c. Validation scalar metrics (best_val_acc, best_val_f1)
val_scalar_records = []
for seed, users in val_scalar_data.items():
    for user, metrics in users.items():
        val_scalar_records.append({
            'seed': seed,
            'test_user': user,
            'best_val_acc': metrics[0],
            'best_val_f1': metrics[1],
        })
val_scalar_df = pd.DataFrame(val_scalar_records)

# =============================================================================
# 3. VERIFICATION: Sum of supports per fold must match known totals
# =============================================================================

print("=" * 70)
print("VERIFICATION: Support sums per (seed, user) — TEST set")
print("=" * 70)
test_support_summary = test_df.groupby(['seed', 'test_user'])['support'].sum().reset_index()
test_support_summary.columns = ['seed', 'test_user', 'sum_support']
print(test_support_summary.to_string(index=False))

expected_test_totals = {
    (42, 'user1'): 38589, (42, 'user2'): 32238, (42, 'user3'): 28025, (42, 'user5'): 37406,
    (43, 'user1'): 38589, (43, 'user2'): 32238, (43, 'user3'): 28025, (43, 'user5'): 37406,
    (44, 'user1'): 38589, (44, 'user2'): 32238, (44, 'user3'): 28025, (44, 'user5'): 37406,
}
print("\nChecking TEST totals...")
all_match_test = True
for (s, u), total in expected_test_totals.items():
    computed = test_support_summary[(test_support_summary['seed']==s) & (test_support_summary['test_user']==u)]['sum_support'].values[0]
    if computed != total:
        print(f"  MISMATCH for seed {s}, user {u}: computed {computed} vs expected {total}")
        all_match_test = False
if all_match_test:
    print("  All TEST support sums match the expected totals ✓")

print("\n" + "=" * 70)
print("VERIFICATION: Support sums per (seed, user) — VALIDATION set")
print("=" * 70)
val_support_summary = val_df.groupby(['seed', 'test_user'])['support'].sum().reset_index()
val_support_summary.columns = ['seed', 'test_user', 'sum_support']
print(val_support_summary.to_string(index=False))

expected_val_totals = {
    (42, 'user1'): 16218, (42, 'user2'): 16967, (42, 'user3'): 16205, (42, 'user5'): 16142,
    (43, 'user1'): 16201, (43, 'user2'): 16760, (43, 'user3'): 16012, (43, 'user5'): 15965,
    (44, 'user1'): 16080, (44, 'user2'): 16977, (44, 'user3'): 16087, (44, 'user5'): 16037,
}
print("\nChecking VALIDATION totals...")
all_match_val = True
for (s, u), total in expected_val_totals.items():
    computed = val_support_summary[(val_support_summary['seed']==s) & (val_support_summary['test_user']==u)]['sum_support'].values[0]
    if computed != total:
        print(f"  MISMATCH for seed {s}, user {u}: computed {computed} vs expected {total}")
        all_match_val = False
if all_match_val:
    print("  All VALIDATION support sums match the expected totals ✓")
print("=" * 70)

# =============================================================================
# 4. SAVE TO CSV FOR FUTURE USE
# =============================================================================
test_df.to_csv('test_classification_reports.csv', index=False)
val_df.to_csv('val_classification_reports.csv', index=False)
val_scalar_df.to_csv('val_metrics.csv', index=False)
print("\nDataFrames saved to CSV files:")
print("  - test_classification_reports.csv")
print("  - val_classification_reports.csv")
print("  - val_metrics.csv (scalar best metrics)")

# =============================================================================
# 5. CALCULATE PER-USER AND OVERALL STATISTICS (as requested)
# =============================================================================
print("\n" + "=" * 70)
print("CALCULATED STATISTICS")
print("=" * 70)

# 5a. Test set statistics
# We need accuracy per seed/user. I'll hard-code it from the logs.
test_accuracies = {
    42: {'user1': 0.9391, 'user2': 0.9693, 'user3': 0.9724, 'user5': 0.9820},
    43: {'user1': 0.9281, 'user2': 0.9735, 'user3': 0.9772, 'user5': 0.9761},
    44: {'user1': 0.9377, 'user2': 0.9842, 'user3': 0.9648, 'user5': 0.9846},
}
test_acc_records = []
for s, users in test_accuracies.items():
    for u, acc in users.items():
        test_acc_records.append({'seed': s, 'test_user': u, 'accuracy': acc})
test_acc_df = pd.DataFrame(test_acc_records)

# Compute macro F1 per (seed, user) from the per-class F1 scores
test_f1_per_fold = test_df.groupby(['seed', 'test_user'])['f1'].mean().reset_index()
test_f1_per_fold.columns = ['seed', 'test_user', 'macro_f1']

# Merge accuracy and macro F1
test_merged = pd.merge(test_acc_df, test_f1_per_fold, on=['seed', 'test_user'])

# Per-user summary (mean ± std across 3 seeds)
test_per_user = test_merged.groupby('test_user').agg(
    acc_mean=('accuracy', 'mean'),
    acc_std=('accuracy', 'std'),
    f1_mean=('macro_f1', 'mean'),
    f1_std=('macro_f1', 'std'),
).reset_index()

test_per_user['acc_mean_pct'] = test_per_user['acc_mean'] * 100
test_per_user['acc_std_pct'] = test_per_user['acc_std'] * 100
test_per_user['f1_mean_pct'] = test_per_user['f1_mean'] * 100
test_per_user['f1_std_pct'] = test_per_user['f1_std'] * 100

print("\n--- TEST SET: Per-User Averages (± std) ---")
print(test_per_user[['test_user', 'acc_mean_pct', 'acc_std_pct', 'f1_mean_pct', 'f1_std_pct']].to_string(index=False))

# Overall test averages
overall_test_acc = test_merged['accuracy'].mean()
overall_test_acc_std = test_merged['accuracy'].std(ddof=1)
overall_test_f1 = test_merged['macro_f1'].mean()
overall_test_f1_std = test_merged['macro_f1'].std(ddof=1)

print("\n--- TEST SET: Overall Averages (± std) ---")
print(f"Accuracy: {overall_test_acc*100:.4f} ± {overall_test_acc_std*100:.4f} %")
print(f"Macro F1 : {overall_test_f1*100:.4f} ± {overall_test_f1_std*100:.4f} %")

# 5b. Validation set statistics
# We already have best_val_acc and best_val_f1 in val_scalar_df.
# Compute per-user summary
val_per_user = val_scalar_df.groupby('test_user').agg(
    acc_mean=('best_val_acc', 'mean'),
    acc_std=('best_val_acc', 'std'),
    f1_mean=('best_val_f1', 'mean'),
    f1_std=('best_val_f1', 'std'),
).reset_index()

val_per_user['acc_mean_pct'] = val_per_user['acc_mean'] * 100
val_per_user['acc_std_pct'] = val_per_user['acc_std'] * 100
val_per_user['f1_mean_pct'] = val_per_user['f1_mean'] * 100
val_per_user['f1_std_pct'] = val_per_user['f1_std'] * 100

print("\n--- VALIDATION SET: Per-User Averages (± std) ---")
print(val_per_user[['test_user', 'acc_mean_pct', 'acc_std_pct', 'f1_mean_pct', 'f1_std_pct']].to_string(index=False))

# Overall validation averages
overall_val_acc = val_scalar_df['best_val_acc'].mean()
overall_val_acc_std = val_scalar_df['best_val_acc'].std(ddof=1)
overall_val_f1 = val_scalar_df['best_val_f1'].mean()
overall_val_f1_std = val_scalar_df['best_val_f1'].std(ddof=1)

print("\n--- VALIDATION SET: Overall Averages (± std) ---")
print(f"Accuracy: {overall_val_acc*100:.4f} ± {overall_val_acc_std*100:.4f} %")
print(f"Macro F1 : {overall_val_f1*100:.4f} ± {overall_val_f1_std*100:.4f} %")
print("=" * 70)


VERIFICATION: Support sums per (seed, user) — TEST set
 seed test_user  sum_support
   42     user1        38589
   42     user2        32238
   42     user3        28025
   42     user5        37406
   43     user1        38589
   43     user2        32238
   43     user3        28025
   43     user5        37406
   44     user1        38589
   44     user2        32238
   44     user3        28025
   44     user5        37406

Checking TEST totals...
  All TEST support sums match the expected totals ✓

VERIFICATION: Support sums per (seed, user) — VALIDATION set
 seed test_user  sum_support
   42     user1        16218
   42     user2        16967
   42     user3        16205
   42     user5        16142
   43     user1        16201
   43     user2        16760
   43     user3        16012
   43     user5        15965
   44     user1        16080
   44     user2        16977
   44     user3        16087
   44     user5        16037

Checking VALIDATION totals...
  All VALIDATION supp

In [2]:

CLASS_ORDER = ['AUGUST', 'BIG', 'BIRD', 'BOAT', 'COME', 'DRIVER', 'FARMING',
               'FEBRUARY', 'GO', 'GREETINGS', 'OUR', 'READ', 'SMALL', 'TIGER',
               'TRAIN', 'UGLY', 'VAN', 'WHAT', 'WHICH', 'WRITE', 'background']
 
USER_ORDER = ['user1', 'user2', 'user3', 'user5']
 
 
# =============================================================================
# 2. BUILD DATAFRAMES
# =============================================================================
 
def build_dataframes():
    test_records = []
    for seed, users in test_data.items():
        for user, classes in users.items():
            for cls, metrics in classes.items():
                test_records.append({
                    'seed': seed, 'test_user': user, 'class_name': cls,
                    'precision': metrics[0], 'recall': metrics[1],
                    'f1': metrics[2], 'support': metrics[3],
                })
    test_df = pd.DataFrame(test_records)
 
    test_acc_records = [{'seed': s, 'test_user': u, 'accuracy': acc}
                         for s, users in test_accuracies.items()
                         for u, acc in users.items()]
    test_acc_df = pd.DataFrame(test_acc_records)
 
    val_records = [{'seed': s, 'test_user': u, 'best_val_acc': m[0], 'best_val_f1': m[1]}
                    for s, users in val_scalar_data.items()
                    for u, m in users.items()]
    val_df = pd.DataFrame(val_records)
 
    return test_df, test_acc_df, val_df
 
 
# =============================================================================
# 3. VERIFICATION (support sums must match known per-fold totals)
# =============================================================================
 
EXPECTED_TEST_TOTALS = {
    (42, 'user1'): 38589, (42, 'user2'): 32238, (42, 'user3'): 28025, (42, 'user5'): 37406,
    (43, 'user1'): 38589, (43, 'user2'): 32238, (43, 'user3'): 28025, (43, 'user5'): 37406,
    (44, 'user1'): 38589, (44, 'user2'): 32238, (44, 'user3'): 28025, (44, 'user5'): 37406,
}
 
 
def verify_supports(test_df):
    sums = test_df.groupby(['seed', 'test_user'])['support'].sum()
    ok = True
    for (s, u), expected in EXPECTED_TEST_TOTALS.items():
        got = sums.get((s, u))
        if got != expected:
            print(f"  MISMATCH seed={s} user={u}: got {got}, expected {expected}")
            ok = False
    print("Support verification:", "PASSED" if ok else "FAILED")
    return ok
 
 
# =============================================================================
# 4. TABLE 1 - TEST set: per-user Accuracy & Macro-F1 (incl. background)
# =============================================================================
 
def build_table1(test_df, test_acc_df):
    macro_f1 = (test_df.groupby(['seed', 'test_user'])['f1']
                .mean().reset_index().rename(columns={'f1': 'macro_f1'}))
    merged = pd.merge(test_acc_df, macro_f1, on=['seed', 'test_user'])
 
    per_user = (merged.groupby('test_user')
                .agg(acc_mean=('accuracy', 'mean'), acc_std=('accuracy', 'std'),
                     f1_mean=('macro_f1', 'mean'), f1_std=('macro_f1', 'std'))
                .reindex(USER_ORDER).reset_index())
 
    # "Mean (LOUO)" row: mean/std computed across the per-user means
    user_means = merged.groupby('test_user')[['accuracy', 'macro_f1']].mean()
    overall = pd.DataFrame([{
        'test_user': 'Mean (LOUO)',
        'acc_mean': user_means['accuracy'].mean(), 'acc_std': user_means['accuracy'].std(ddof=1),
        'f1_mean': user_means['macro_f1'].mean(), 'f1_std': user_means['macro_f1'].std(ddof=1),
    }])
    table1 = pd.concat([per_user, overall], ignore_index=True)
    for col in ['acc_mean', 'acc_std', 'f1_mean', 'f1_std']:
        table1[col] = table1[col] * 100
    return table1
 
 
# =============================================================================
# 5. TABLE 2 - VALIDATION set: per-user best Accuracy & F1
# =============================================================================
 
def build_table2(val_df):
    per_user = (val_df.groupby('test_user')
                .agg(acc_mean=('best_val_acc', 'mean'), acc_std=('best_val_acc', 'std'),
                     f1_mean=('best_val_f1', 'mean'), f1_std=('best_val_f1', 'std'))
                .reindex(USER_ORDER).reset_index())
 
    user_means = val_df.groupby('test_user')[['best_val_acc', 'best_val_f1']].mean()
    overall = pd.DataFrame([{
        'test_user': 'Mean',
        'acc_mean': user_means['best_val_acc'].mean(), 'acc_std': user_means['best_val_acc'].std(ddof=1),
        'f1_mean': user_means['best_val_f1'].mean(), 'f1_std': user_means['best_val_f1'].std(ddof=1),
    }])
    table2 = pd.concat([per_user, overall], ignore_index=True)
    for col in ['acc_mean', 'acc_std', 'f1_mean', 'f1_std']:
        table2[col] = table2[col] * 100
    return table2
 
 
# =============================================================================
# 6. TABLE 3 - TEST set: per-class F1 only, incl. background
#    (mean/std across all 12 folds: 4 users x 3 seeds)
# =============================================================================
 
def build_table3(test_df):
    per_class = (test_df.groupby('class_name')['f1']
                 .agg(['mean', 'std']).reindex(CLASS_ORDER).reset_index())
    per_class.columns = ['class_name', 'f1_mean', 'f1_std']
    per_class['f1_mean'] *= 100
    per_class['f1_std'] *= 100
 
    macro_row = pd.DataFrame([{
        'class_name': 'Macro avg',
        'f1_mean': per_class['f1_mean'].mean(),
        'f1_std': np.nan,
    }])
    return pd.concat([per_class, macro_row], ignore_index=True)
 
 
# =============================================================================
# 7. PRETTY-PRINT / EXPORT HELPERS
# =============================================================================
 
def print_table1(table1):
    print("\nTABLE 1 - TEST SET: Per-user Accuracy & Macro-F1 (21 classes incl. background)")
    print("mean +/- std over 3 seeds (42, 43, 44)\n")
    print(f"{'Held-out user':<15}{'Accuracy (%)':<18}{'Macro F1 (%)':<18}")
    for _, r in table1.iterrows():
        print(f"{r.test_user:<15}{r.acc_mean:5.4f} +/- {r.acc_std:4.4f}   "
              f"{r.f1_mean:5.4f} +/- {r.f1_std:4.4f}")
 
 
def print_table2(table2):
    print("\nTABLE 2 - VALIDATION SET: Per-user best Accuracy & F1")
    print("mean +/- std over 3 seeds (42, 43, 44)\n")
    print(f"{'Held-out user':<15}{'Accuracy (%)':<18}{'F1 (%)':<18}")
    for _, r in table2.iterrows():
        print(f"{r.test_user:<15}{r.acc_mean:5.4f} +/- {r.acc_std:4.4f}   "
              f"{r.f1_mean:5.4f} +/- {r.f1_std:4.4f}")
 
 
def print_table3(table3):
    print("\nTABLE 3 - TEST SET: Per-class F1 (incl. background)")
    print("mean +/- std across all 12 folds (4 users x 3 seeds)\n")
    print(f"{'Sign class':<14}{'F1 (%)':<14}")
    for _, r in table3.iterrows():
        if pd.isna(r.f1_std):
            print(f"{r.class_name:<14}{r.f1_mean:5.4f}")
        else:
            print(f"{r.class_name:<14}{r.f1_mean:5.4f} +/- {r.f1_std:4.4f}")
 
 
def to_markdown_table1(table1):
    lines = ["| Held-out User | Accuracy (%) | Macro F1 (%) |", "|---|---|---|"]
    for _, r in table1.iterrows():
        label = f"**{r.test_user}**" if r.test_user == 'Mean (LOUO)' else r.test_user
        lines.append(f"| {label} | {r.acc_mean:.4f} ± {r.acc_std:.4f} | {r.f1_mean:.4f} ± {r.f1_std:.4f} |")
    return "\n".join(lines)
 
 
def to_markdown_table2(table2):
    lines = ["| Held-out User | Accuracy (%) | F1 (%) |", "|---|---|---|"]
    for _, r in table2.iterrows():
        label = f"**{r.test_user}**" if r.test_user == 'Mean' else r.test_user
        lines.append(f"| {label} | {r.acc_mean:.4f} ± {r.acc_std:.4f} | {r.f1_mean:.4f} ± {r.f1_std:.4f} |")
    return "\n".join(lines)
 
 
def to_markdown_table3(table3):
    lines = ["| Sign Class | F1-score (%) |", "|---|---|"]
    for _, r in table3.iterrows():
        label = f"**{r.class_name}**" if r.class_name == 'Macro avg' else r.class_name
        if pd.isna(r.f1_std):
            lines.append(f"| {label} | {r.f1_mean:.4f} |")
        else:
            lines.append(f"| {label} | {r.f1_mean:.4f} ± {r.f1_std:.4f} |")
    return "\n".join(lines)
 
 
# =============================================================================
# 8. MAIN
# =============================================================================
 
if __name__ == "__main__":
    test_df, test_acc_df, val_df = build_dataframes()
 
    print("=" * 70)
    verify_supports(test_df)
    print("=" * 70)
 
    table1 = build_table1(test_df, test_acc_df)
    table2 = build_table2(val_df)
    table3 = build_table3(test_df)
 
    print_table1(table1)
    print_table2(table2)
    print_table3(table3)
 
    # Save CSVs (raw numeric, for further processing / plotting)
    table1.to_csv("thesis_table1_test_per_user.csv", index=False)
    table2.to_csv("thesis_table2_val_per_user.csv", index=False)
    table3.to_csv("thesis_table3_test_per_class_f1.csv", index=False)
 
    # Save Markdown (ready to paste into a thesis / README / Overleaf via
    # a markdown-to-latex table converter)
    with open("thesis_tables.md", "w") as f:
        f.write("## Table 1 - TEST set: per-user Accuracy & Macro-F1\n\n")
        f.write(to_markdown_table1(table1))
        f.write("\n\n## Table 2 - VALIDATION set: per-user Accuracy & F1\n\n")
        f.write(to_markdown_table2(table2))
        f.write("\n\n## Table 3 - TEST set: per-class F1 (incl. background)\n\n")
        f.write(to_markdown_table3(table3))
        f.write("\n")
 
    print("\n" + "=" * 70)
    print("Saved:")
    print("  thesis_table1_test_per_user.csv")
    print("  thesis_table2_val_per_user.csv")
    print("  thesis_table3_test_per_class_f1.csv")
    print("  thesis_tables.md   (ready-to-paste Markdown versions of all 3 tables)")
    print("=" * 70)

Support verification: PASSED

TABLE 1 - TEST SET: Per-user Accuracy & Macro-F1 (21 classes incl. background)
mean +/- std over 3 seeds (42, 43, 44)

Held-out user  Accuracy (%)      Macro F1 (%)      
user1          93.4967 +/- 0.5988   85.6190 +/- 2.2743
user2          97.5667 +/- 0.7683   93.5556 +/- 1.9751
user3          97.1467 +/- 0.6252   90.8413 +/- 1.5484
user5          98.0900 +/- 0.4355   93.4762 +/- 1.2626
Mean (LOUO)    96.5750 +/- 2.0882   90.8730 +/- 3.7228

TABLE 2 - VALIDATION SET: Per-user best Accuracy & F1
mean +/- std over 3 seeds (42, 43, 44)

Held-out user  Accuracy (%)      F1 (%)            
user1          99.6573 +/- 0.1872   99.1204 +/- 0.4190
user2          99.4976 +/- 0.1054   98.8575 +/- 0.2159
user3          99.3580 +/- 0.2306   98.5592 +/- 0.6197
user5          99.4763 +/- 0.2008   98.7600 +/- 0.4967
Mean           99.4973 +/- 0.1231   98.8243 +/- 0.2332

TABLE 3 - TEST SET: Per-class F1 (incl. background)
mean +/- std across all 12 folds (4 users x 3 see

In [7]:
# =============================================================================
# 2. BUILD DATAFRAMES
# =============================================================================
 
def build_dataframes():
    test_records = []
    for seed, users in test_data.items():
        for user, classes in users.items():
            for cls, metrics in classes.items():
                test_records.append({
                    'seed': seed, 'test_user': user, 'class_name': cls,
                    'precision': metrics[0], 'recall': metrics[1],
                    'f1': metrics[2], 'support': metrics[3],
                })
    test_df = pd.DataFrame(test_records)
 
    test_acc_records = [{'seed': s, 'test_user': u, 'accuracy': acc}
                         for s, users in test_accuracies.items()
                         for u, acc in users.items()]
    test_acc_df = pd.DataFrame(test_acc_records)
 
    val_records = [{'seed': s, 'test_user': u, 'best_val_acc': m[0], 'best_val_f1': m[1]}
                    for s, users in val_scalar_data.items()
                    for u, m in users.items()]
    val_df = pd.DataFrame(val_records)
 
    val_class_records = []
    for seed, users in val_classification_data.items():
        for user, classes in users.items():
            for cls, metrics in classes.items():
                val_class_records.append({
                    'seed': seed, 'test_user': user, 'class_name': cls,
                    'precision': metrics[0], 'recall': metrics[1],
                    'f1': metrics[2], 'support': metrics[3],
                })
    val_class_df = pd.DataFrame(val_class_records)
 
    return test_df, test_acc_df, val_df, val_class_df
 
 
# =============================================================================
# 3. VERIFICATION (support sums must match known per-fold totals)
# =============================================================================
 
EXPECTED_TEST_TOTALS = {
    (42, 'user1'): 38589, (42, 'user2'): 32238, (42, 'user3'): 28025, (42, 'user5'): 37406,
    (43, 'user1'): 38589, (43, 'user2'): 32238, (43, 'user3'): 28025, (43, 'user5'): 37406,
    (44, 'user1'): 38589, (44, 'user2'): 32238, (44, 'user3'): 28025, (44, 'user5'): 37406,
}
 
 
def verify_supports(test_df):
    sums = test_df.groupby(['seed', 'test_user'])['support'].sum()
    ok = True
    for (s, u), expected in EXPECTED_TEST_TOTALS.items():
        got = sums.get((s, u))
        if got != expected:
            print(f"  MISMATCH seed={s} user={u}: got {got}, expected {expected}")
            ok = False
    print("Support verification:", "PASSED" if ok else "FAILED")
    return ok
 
 
# =============================================================================
# 4. TABLE 1 - TEST set: per-user Accuracy & Macro-F1 (incl. background)
# =============================================================================
 
def build_table1(test_df, test_acc_df):
    macro_f1 = (test_df.groupby(['seed', 'test_user'])['f1']
                .mean().reset_index().rename(columns={'f1': 'macro_f1'}))
    merged = pd.merge(test_acc_df, macro_f1, on=['seed', 'test_user'])
 
    per_user = (merged.groupby('test_user')
                .agg(acc_mean=('accuracy', 'mean'), acc_std=('accuracy', 'std'),
                     f1_mean=('macro_f1', 'mean'), f1_std=('macro_f1', 'std'))
                .reindex(USER_ORDER).reset_index())
 
    # "Mean (LOUO)" row: mean/std computed across the per-user means
    user_means = merged.groupby('test_user')[['accuracy', 'macro_f1']].mean()
    overall = pd.DataFrame([{
        'test_user': 'Mean (LOUO)',
        'acc_mean': user_means['accuracy'].mean(), 'acc_std': user_means['accuracy'].std(ddof=1),
        'f1_mean': user_means['macro_f1'].mean(), 'f1_std': user_means['macro_f1'].std(ddof=1),
    }])
    table1 = pd.concat([per_user, overall], ignore_index=True)
    for col in ['acc_mean', 'acc_std', 'f1_mean', 'f1_std']:
        table1[col] = table1[col] * 100
    return table1
 
 
# =============================================================================
# 5. TABLE 2 - VALIDATION set: per-user best Accuracy & F1
# =============================================================================
 
def build_table2(val_df):
    per_user = (val_df.groupby('test_user')
                .agg(acc_mean=('best_val_acc', 'mean'), acc_std=('best_val_acc', 'std'),
                     f1_mean=('best_val_f1', 'mean'), f1_std=('best_val_f1', 'std'))
                .reindex(USER_ORDER).reset_index())
 
    user_means = val_df.groupby('test_user')[['best_val_acc', 'best_val_f1']].mean()
    overall = pd.DataFrame([{
        'test_user': 'Mean',
        'acc_mean': user_means['best_val_acc'].mean(), 'acc_std': user_means['best_val_acc'].std(ddof=1),
        'f1_mean': user_means['best_val_f1'].mean(), 'f1_std': user_means['best_val_f1'].std(ddof=1),
    }])
    table2 = pd.concat([per_user, overall], ignore_index=True)
    for col in ['acc_mean', 'acc_std', 'f1_mean', 'f1_std']:
        table2[col] = table2[col] * 100
    return table2
 
 
# =============================================================================
# 6. TABLE 3 - TEST set: per-class F1 only, incl. background
#    (mean/std across all 12 folds: 4 users x 3 seeds)
# =============================================================================
 
def build_per_class_f1_table(class_df):
    """Works for either the TEST or VALIDATION per-class dataframe."""
    per_class = (class_df.groupby('class_name')['f1']
                 .agg(['mean', 'std']).reindex(CLASS_ORDER).reset_index())
    per_class.columns = ['class_name', 'f1_mean', 'f1_std']
    per_class['f1_mean'] *= 100
    per_class['f1_std'] *= 100
 
    macro_row = pd.DataFrame([{
        'class_name': 'Macro avg',
        'f1_mean': per_class['f1_mean'].mean(),
        'f1_std': np.nan,
    }])
    return pd.concat([per_class, macro_row], ignore_index=True)
 
 
# =============================================================================
# 7. PRETTY-PRINT / EXPORT HELPERS
# =============================================================================
 
def print_table1(table1):
    print("\nTABLE 1 - TEST SET: Per-user Accuracy & Macro-F1 (21 classes incl. background)")
    print("mean +/- std over 3 seeds (42, 43, 44)\n")
    print(f"{'Held-out user':<15}{'Accuracy (%)':<18}{'Macro F1 (%)':<18}")
    for _, r in table1.iterrows():
        print(f"{r.test_user:<15}{r.acc_mean:5.4f} +/- {r.acc_std:4.4f}   "
              f"{r.f1_mean:5.4f} +/- {r.f1_std:4.4f}")
 
 
def print_table2(table2):
    print("\nTABLE 2 - VALIDATION SET: Per-user best Accuracy & F1")
    print("mean +/- std over 3 seeds (42, 43, 44)\n")
    print(f"{'Held-out user':<15}{'Accuracy (%)':<18}{'F1 (%)':<18}")
    for _, r in table2.iterrows():
        print(f"{r.test_user:<15}{r.acc_mean:5.4f} +/- {r.acc_std:4.4f}   "
              f"{r.f1_mean:5.4f} +/- {r.f1_std:4.4f}")
 
 
def print_per_class_table(table, title):
    print(f"\n{title}")
    print("mean +/- std across all 12 folds (4 users x 3 seeds)\n")
    print(f"{'Sign class':<14}{'F1 (%)':<14}")
    for _, r in table.iterrows():
        if pd.isna(r.f1_std):
            print(f"{r.class_name:<14}{r.f1_mean:5.4f}")
        else:
            print(f"{r.class_name:<14}{r.f1_mean:5.4f} +/- {r.f1_std:4.4f}")
 
 
def to_markdown_table1(table1):
    lines = ["| Held-out User | Accuracy (%) | Macro F1 (%) |", "|---|---|---|"]
    for _, r in table1.iterrows():
        label = f"**{r.test_user}**" if r.test_user == 'Mean (LOUO)' else r.test_user
        lines.append(f"| {label} | {r.acc_mean:.4f} ± {r.acc_std:.4f} | {r.f1_mean:.4f} ± {r.f1_std:.4f} |")
    return "\n".join(lines)
 
 
def to_markdown_table2(table2):
    lines = ["| Held-out User | Accuracy (%) | F1 (%) |", "|---|---|---|"]
    for _, r in table2.iterrows():
        label = f"**{r.test_user}**" if r.test_user == 'Mean' else r.test_user
        lines.append(f"| {label} | {r.acc_mean:.4f} ± {r.acc_std:.4f} | {r.f1_mean:.4f} ± {r.f1_std:.4f} |")
    return "\n".join(lines)
 
 
def to_markdown_per_class_table(table):
    lines = ["| Sign Class | F1-score (%) |", "|---|---|"]
    for _, r in table.iterrows():
        label = f"**{r.class_name}**" if r.class_name == 'Macro avg' else r.class_name
        if pd.isna(r.f1_std):
            lines.append(f"| {label} | {r.f1_mean:.4f} |")
        else:
            lines.append(f"| {label} | {r.f1_mean:.4f} ± {r.f1_std:.4f} |")
    return "\n".join(lines)
 
 
# =============================================================================
# 8. MAIN
# =============================================================================
 
if __name__ == "__main__":
    test_df, test_acc_df, val_df, val_class_df = build_dataframes()
 
    print("=" * 70)
    verify_supports(test_df)
    print("=" * 70)
 
    table1 = build_table1(test_df, test_acc_df)
    table2 = build_table2(val_df)
    table3 = build_per_class_f1_table(test_df)       # TEST per-class F1
    table4 = build_per_class_f1_table(val_class_df)  # VAL per-class F1
 
    print_table1(table1)
    print_table2(table2)
    print_per_class_table(table3, "TABLE 3 - TEST SET: Per-class F1 (incl. background)")
    print_per_class_table(table4, "TABLE 4 - VALIDATION SET: Per-class F1 (incl. background)")
 
    # Save CSVs (raw numeric, for further processing / plotting)
    table1.to_csv("thesis_table1_test_per_user.csv", index=False)
    table2.to_csv("thesis_table2_val_per_user.csv", index=False)
    table3.to_csv("thesis_table3_test_per_class_f1.csv", index=False)
    table4.to_csv("thesis_table4_val_per_class_f1.csv", index=False)
 
    # Save Markdown (ready to paste into a thesis / README / Overleaf via
    # a markdown-to-latex table converter)
    with open("thesis_tables.md", "w") as f:
        f.write("## Table 1 - TEST set: per-user Accuracy & Macro-F1\n\n")
        f.write(to_markdown_table1(table1))
        f.write("\n\n## Table 2 - VALIDATION set: per-user Accuracy & F1\n\n")
        f.write(to_markdown_table2(table2))
        f.write("\n\n## Table 3 - TEST set: per-class F1 (incl. background)\n\n")
        f.write(to_markdown_per_class_table(table3))
        f.write("\n\n## Table 4 - VALIDATION set: per-class F1 (incl. background)\n\n")
        f.write(to_markdown_per_class_table(table4))
        f.write("\n")
 
    print("\n" + "=" * 70)
    print("Saved:")
    print("  thesis_table1_test_per_user.csv")
    print("  thesis_table2_val_per_user.csv")
    print("  thesis_table3_test_per_class_f1.csv")
    print("  thesis_table4_val_per_class_f1.csv")
    print("  thesis_tables.md   (ready-to-paste Markdown versions of all 4 tables)")
    print("=" * 70)

Support verification: PASSED

TABLE 1 - TEST SET: Per-user Accuracy & Macro-F1 (21 classes incl. background)
mean +/- std over 3 seeds (42, 43, 44)

Held-out user  Accuracy (%)      Macro F1 (%)      
user1          93.4967 +/- 0.5988   85.6190 +/- 2.2743
user2          97.5667 +/- 0.7683   93.5556 +/- 1.9751
user3          97.1467 +/- 0.6252   90.8413 +/- 1.5484
user5          98.0900 +/- 0.4355   93.4762 +/- 1.2626
Mean (LOUO)    96.5750 +/- 2.0882   90.8730 +/- 3.7228

TABLE 2 - VALIDATION SET: Per-user best Accuracy & F1
mean +/- std over 3 seeds (42, 43, 44)

Held-out user  Accuracy (%)      F1 (%)            
user1          99.6573 +/- 0.1872   99.1204 +/- 0.4190
user2          99.4976 +/- 0.1054   98.8575 +/- 0.2159
user3          99.3580 +/- 0.2306   98.5592 +/- 0.6197
user5          99.4763 +/- 0.2008   98.7600 +/- 0.4967
Mean           99.4973 +/- 0.1231   98.8243 +/- 0.2332

TABLE 3 - TEST SET: Per-class F1 (incl. background)
mean +/- std across all 12 folds (4 users x 3 see

In [5]:
merged = pd.merge(test_df, test_acc_df, on=['seed', 'test_user'])

In [8]:
macro_f1 = (test_df.groupby(['seed', 'test_user'])['f1']
                .mean().reset_index().rename(columns={'f1': 'macro_f1'}))

In [9]:
macro_f1

,seed,test_user,macro_f1
0,42,user1,0.870952
1,42,user2,0.919524
2,42,user3,0.920476
3,42,user5,0.935238
4,43,user1,0.830000
5,43,user2,0.929524
6,43,user3,0.890952
7,43,user5,0.921905
8,44,user1,0.867619
9,44,user2,0.957619


In [3]:
# """
# ================================================================================
# SIGN LANGUAGE RECOGNITION - EVENT-LEVEL (SEGMENT) METRICS AGGREGATION
# ================================================================================
# Run locally with: python event_level_tables.py
# Requires: pandas, numpy

# This complements thesis_tables.py (which handles WINDOW-level classification
# reports). This script handles EVENT-level metrics: for each ground-truth sign
# INSTANCE (not window), was it detected, was it correctly labeled, how well
# did the predicted segment overlap the true one, and how fast was it caught.

# WHY THIS IS A SEPARATE TABLE FROM THE WINDOW-LEVEL REPORT:
#   Window-level P/R/F1 tells you about per-frame classification accuracy.
#   Event-level metrics tell you whether each *sign occurrence* was correctly
#   detected as a discrete event, in roughly the right place - a different
#   and complementary question, standard in gesture/action-segmentation work.

# HOW TO USE THIS SCRIPT:
#   1. For each (seed, held_out_user) fold, paste the per-class event-level
#      block from your log into EVENT_DATA below, following the same
#      structure as the one example fold already filled in.
#   2. Do the same for the "Mean WER / Median WER / Std WER" summary line
#      into WER_DATA.
#   3. Run the script - it produces:
#        Table 5: per-class Detection/Misclass/False-Pos Rate + Jaccard,
#                 pooled across all folds (mean +/- std, computed two ways
#                 - see note below).
#        Table 6: per-class Avg Latency (frames), pooled across all folds.
#        Table 7: per-user WER summary (mean +/- std over seeds) + overall.

# NOTE ON POOLING vs NAIVE AVERAGING (Table 5):
#   Your per-fold GT counts are small (often 12-16 instances per class).
#   Two valid ways to get a cross-fold class average:
#     (a) "Macro" - average the per-fold RATES directly (each fold weighted
#         equally regardless of GT count). This is what mean/std over rows
#         gives you - simplest, matches how you've built every other table.
#     (b) "Pooled" - sum the raw counts (Detected, Misclassified, FP, GT)
#         across all folds first, THEN compute the rate once. This is more
#         statistically sound when per-fold denominators are small and
#         uneven, because a fold with 16 GT instances gets proportionally
#         more say than one with 12.
#   This script reports BOTH so you can choose / justify your choice in the
#   methodology section. For a thesis, the pooled version is usually the
#   more defensible headline number; the macro (mean+/-std) version is
#   useful for showing fold-to-fold stability.
# ================================================================================
# """

# import pandas as pd
# import numpy as np

# pd.set_option("display.width", 120)

# CLASS_ORDER = ['AUGUST', 'BIG', 'BIRD', 'BOAT', 'COME', 'DRIVER', 'FARMING',
#                'FEBRUARY', 'GO', 'GREETINGS', 'OUR', 'READ', 'SMALL', 'TIGER',
#                'TRAIN', 'UGLY', 'VAN', 'WHAT', 'WHICH', 'WRITE']
# # Note: no 'background' here - event-level detection is defined per sign
# # instance, so there is no "background instance" to detect.

# USER_ORDER = ['user1', 'user2', 'user3', 'user5']

# # =============================================================================
# # 1. RAW DATA - fill in one block per (seed, held_out_user) fold.
# #    Structure: EVENT_DATA[seed][user][class_name] = {
# #        'gt': int,              # number of ground-truth instances
# #        'detected': int,        # correctly detected & correctly labeled
# #        'misclassified': int,   # detected but wrong label
# #        'fp': int,              # false-positive detections (no matching GT)
# #        'latency': float,       # avg latency in frames (only over TP instances)
# #        'jaccard': float,       # avg IoU (only over TP instances)
# #    }
# #    Derive 'detected' from Detection Rate x GT (rounded), 'misclassified'
# #    from Misclass Rate x GT, 'fp' directly from the "X FP / Y GT" line.
# # =============================================================================

# EVENT_DATA = {
#     # >>> REPLACE 'SEED' and 'USER' with the actual seed (42/43/44) and
#     #     held-out user (user1/user2/user3/user5) this log block came from <<<
#     'SEED': {
#         'USER': {
#             'AUGUST':    {'gt': 13, 'detected': 11, 'misclassified': 0, 'fp': 3,  'latency': 22.18, 'jaccard': 0.4713},
#             'BIG':       {'gt': 16, 'detected': 14, 'misclassified': 1, 'fp': 1,  'latency': 28.00, 'jaccard': 0.4401},
#             'BIRD':      {'gt': 13, 'detected': 13, 'misclassified': 0, 'fp': 0,  'latency': 24.23, 'jaccard': 0.6808},
#             'BOAT':      {'gt': 13, 'detected': 13, 'misclassified': 0, 'fp': 2,  'latency': 16.62, 'jaccard': 0.5850},
#             'COME':      {'gt': 14, 'detected': 6,  'misclassified': 3, 'fp': 0,  'latency': 25.17, 'jaccard': 0.1952},
#             'DRIVER':    {'gt': 13, 'detected': 13, 'misclassified': 0, 'fp': 3,  'latency': 30.00, 'jaccard': 0.6340},
#             'FARMING':   {'gt': 16, 'detected': 15, 'misclassified': 1, 'fp': 0,  'latency': 34.67, 'jaccard': 0.6120},
#             'FEBRUARY':  {'gt': 13, 'detected': 13, 'misclassified': 0, 'fp': 2,  'latency': 17.31, 'jaccard': 0.6219},
#             'GO':        {'gt': 15, 'detected': 9,  'misclassified': 1, 'fp': 4,  'latency': 27.22, 'jaccard': 0.2649},
#             'GREETINGS': {'gt': 14, 'detected': 10, 'misclassified': 4, 'fp': 0,  'latency': 29.80, 'jaccard': 0.3823},
#             'OUR':       {'gt': 14, 'detected': 13, 'misclassified': 0, 'fp': 14, 'latency': 23.54, 'jaccard': 0.2666},
#             'READ':      {'gt': 15, 'detected': 14, 'misclassified': 0, 'fp': 2,  'latency': 22.50, 'jaccard': 0.6032},
#             'SMALL':     {'gt': 16, 'detected': 16, 'misclassified': 0, 'fp': 0,  'latency': 16.44, 'jaccard': 0.6829},
#             'TIGER':     {'gt': 13, 'detected': 12, 'misclassified': 1, 'fp': 2,  'latency': 17.42, 'jaccard': 0.4190},
#             'TRAIN':     {'gt': 12, 'detected': 10, 'misclassified': 1, 'fp': 2,  'latency': 28.40, 'jaccard': 0.4498},
#             'UGLY':      {'gt': 13, 'detected': 13, 'misclassified': 0, 'fp': 3,  'latency': 19.69, 'jaccard': 0.5287},
#             'VAN':       {'gt': 16, 'detected': 16, 'misclassified': 0, 'fp': 0,  'latency': 19.75, 'jaccard': 0.6518},
#             'WHAT':      {'gt': 14, 'detected': 5,  'misclassified': 0, 'fp': 11, 'latency': 21.00, 'jaccard': 0.1492},
#             'WHICH':     {'gt': 13, 'detected': 9,  'misclassified': 1, 'fp': 13, 'latency': 23.67, 'jaccard': 0.1966},
#             'WRITE':     {'gt': 13, 'detected': 13, 'misclassified': 0, 'fp': 2,  'latency': 14.08, 'jaccard': 0.5315},
#         },
#         # >>> add 'user2': {...}, 'user3': {...}, 'user5': {...} here <<<
#     },
#     # >>> add 43: {...}, 44: {...} here, each with all 4 users <<<
# }

# # WER summary line per (seed, user) fold, straight from the log.
# WER_DATA = {
#     'SEED': {
#         'USER': {'mean_wer': 0.2094, 'median_wer': 0.0000, 'std_wer': 0.3416},
#     },
#     # >>> add the other 11 folds here <<<
# }


# # =============================================================================
# # 2. BUILD DATAFRAME (long format: one row per seed x user x class)
# # =============================================================================

# def build_event_df():
#     records = []
#     for seed, users in EVENT_DATA.items():
#         for user, classes in users.items():
#             for cls, m in classes.items():
#                 gt = m['gt']
#                 records.append({
#                     'seed': seed, 'test_user': user, 'class_name': cls,
#                     'gt': gt,
#                     'detected': m['detected'],
#                     'misclassified': m['misclassified'],
#                     'fp': m['fp'],
#                     'latency': m['latency'],
#                     'jaccard': m['jaccard'],
#                     'detection_rate': m['detected'] / gt if gt else np.nan,
#                     'misclass_rate': m['misclassified'] / gt if gt else np.nan,
#                     'fp_rate': m['fp'] / gt if gt else np.nan,
#                 })
#     return pd.DataFrame(records)


# def build_wer_df():
#     records = []
#     for seed, users in WER_DATA.items():
#         for user, m in users.items():
#             records.append({'seed': seed, 'test_user': user, **m})
#     return pd.DataFrame(records)


# # =============================================================================
# # 3. TABLE 5 - per-class Detection/Misclass/FalsePos rate + Jaccard
# #    Reports BOTH the macro (mean of per-fold rates) and pooled
# #    (rate of summed counts) versions - see note at top of file.
# # =============================================================================

# def build_table5(event_df):
#     # Macro: average the per-fold rates directly
#     macro = (event_df.groupby('class_name')[['detection_rate', 'misclass_rate', 'fp_rate', 'jaccard']]
#              .agg(['mean', 'std']).reindex(CLASS_ORDER))

#     # Pooled: sum raw counts across folds, then compute one rate per class
#     pooled = event_df.groupby('class_name').agg(
#         gt_sum=('gt', 'sum'), detected_sum=('detected', 'sum'),
#         misclassified_sum=('misclassified', 'sum'), fp_sum=('fp', 'sum'),
#     ).reindex(CLASS_ORDER)
#     pooled['detection_rate_pooled'] = pooled['detected_sum'] / pooled['gt_sum']
#     pooled['misclass_rate_pooled'] = pooled['misclassified_sum'] / pooled['gt_sum']
#     pooled['fp_rate_pooled'] = pooled['fp_sum'] / pooled['gt_sum']

#     table5 = pd.DataFrame({
#         'class_name': CLASS_ORDER,
#         'detection_rate_mean': macro[('detection_rate', 'mean')].values,
#         'detection_rate_std': macro[('detection_rate', 'std')].values,
#         'detection_rate_pooled': pooled['detection_rate_pooled'].values,
#         'misclass_rate_mean': macro[('misclass_rate', 'mean')].values,
#         'misclass_rate_std': macro[('misclass_rate', 'std')].values,
#         'misclass_rate_pooled': pooled['misclass_rate_pooled'].values,
#         'fp_rate_mean': macro[('fp_rate', 'mean')].values,
#         'fp_rate_std': macro[('fp_rate', 'std')].values,
#         'fp_rate_pooled': pooled['fp_rate_pooled'].values,
#         'jaccard_mean': macro[('jaccard', 'mean')].values,
#         'jaccard_std': macro[('jaccard', 'std')].values,
#     })
#     return table5


# # =============================================================================
# # 4. TABLE 6 - per-class Avg Latency (frames), pooled across folds,
# #    weighted by number of detected (TP) instances per fold since latency
# #    is only defined for instances that were actually detected.
# # =============================================================================

# def build_table6(event_df):
#     def weighted_latency(g):
#         w = g['detected']
#         if w.sum() == 0:
#             return np.nan
#         return np.average(g['latency'], weights=w)

#     macro = event_df.groupby('class_name')['latency'].agg(['mean', 'std']).reindex(CLASS_ORDER)
#     pooled = event_df.groupby('class_name').apply(weighted_latency, include_groups=False).reindex(CLASS_ORDER)

#     table6 = pd.DataFrame({
#         'class_name': CLASS_ORDER,
#         'latency_mean': macro['mean'].values,
#         'latency_std': macro['std'].values,
#         'latency_pooled_weighted': pooled.values,
#     })
#     return table6


# # =============================================================================
# # 5. TABLE 7 - WER summary, per-user (mean +/- std over seeds) + overall
# # =============================================================================

# def build_table7(wer_df):
#     per_user = (wer_df.groupby('test_user')['mean_wer']
#                 .agg(['mean', 'std']).reindex(USER_ORDER).reset_index())
#     per_user.columns = ['test_user', 'wer_mean', 'wer_std']

#     user_means = wer_df.groupby('test_user')['mean_wer'].mean()
#     overall = pd.DataFrame([{
#         'test_user': 'Mean (LOUO)',
#         'wer_mean': user_means.mean(),
#         'wer_std': user_means.std(ddof=1),
#     }])
#     return pd.concat([per_user, overall], ignore_index=True)


# # =============================================================================
# # 6. PRINT HELPERS
# # =============================================================================

# def print_table5(table5):
#     print("\nTABLE 5 - EVENT-LEVEL: Detection Rate / Misclass Rate / False Pos Rate / Jaccard")
#     print("macro = mean+/-std of per-fold rates | pooled = rate from summed counts\n")
#     for _, r in table5.iterrows():
#         print(f"{r.class_name:<11} "
#               f"DR: {r.detection_rate_mean:.4f}+/-{r.detection_rate_std:.4f} (pooled {r.detection_rate_pooled:.4f})  "
#               f"MC: {r.misclass_rate_mean:.4f}+/-{r.misclass_rate_std:.4f} (pooled {r.misclass_rate_pooled:.4f})  "
#               f"FP: {r.fp_rate_mean:.4f}+/-{r.fp_rate_std:.4f} (pooled {r.fp_rate_pooled:.4f})  "
#               f"Jac: {r.jaccard_mean:.4f}+/-{r.jaccard_std:.4f}")


# def print_table6(table6):
#     print("\nTABLE 6 - EVENT-LEVEL: Avg Latency (frames)")
#     print("macro = mean+/-std of per-fold latency | pooled = TP-count-weighted average\n")
#     for _, r in table6.iterrows():
#         print(f"{r.class_name:<11} {r.latency_mean:6.4f} +/- {r.latency_std:5.4f}   (pooled: {r.latency_pooled_weighted:6.4f})")


# def print_table7(table7):
#     print("\nTABLE 7 - WER summary, per held-out user (mean +/- std over 3 seeds)\n")
#     for _, r in table7.iterrows():
#         print(f"{r.test_user:<15} {r.wer_mean:.4f} +/- {r.wer_std:.4f}")


# # =============================================================================
# # 7. MAIN
# # =============================================================================

# if __name__ == "__main__":
#     event_df = build_event_df()
#     wer_df = build_wer_df()

#     n_folds = event_df[['seed', 'test_user']].drop_duplicates().shape[0]
#     print(f"Loaded {n_folds} fold(s) of event-level data.")
#     if n_folds < 12:
#         print("NOTE: You have fewer than 12 folds (3 seeds x 4 users) loaded.")
#         print("      Std values will be NaN or unreliable until all folds are added.")
#         print("      Edit EVENT_DATA and WER_DATA at the top of this file.")

#     table5 = build_table5(event_df)
#     table6 = build_table6(event_df)
#     table7 = build_table7(wer_df)

#     print_table5(table5)
#     print_table6(table6)
#     print_table7(table7)

#     table5.to_csv("thesis_table5_event_level_rates.csv", index=False)
#     table6.to_csv("thesis_table6_event_level_latency.csv", index=False)
#     table7.to_csv("thesis_table7_wer_summary.csv", index=False)
#     print("\nSaved: thesis_table5_event_level_rates.csv, "
#           "thesis_table6_event_level_latency.csv, thesis_table7_wer_summary.csv")